# CircuitGraft — Cloud S4 DOMAIN circuit (drifted model)

**Purpose:** discover the domain circuit D on the drifted model θ_D with
CircuitKIT EAP-IG, and push every artifact to the private HF Hub.

**PRECONDITION — do not start this before:** the local S2 has finished
(~01:00-01:45) and `scripts/upload_drift_to_hf.py` has pushed the merged
drifted checkpoint to `dubeydevansh/circuitgraft-phase4-cloud`. Cell 5 checks
this and stops the run if the checkpoint is not there.

## HOW TO RUN

1. Right sidebar: **Accelerator → GPU T4 x2** (or P100), **Internet → ON**.
2. **Add-ons → Secrets**: `HF_TOKEN` attached to this notebook.
3. **Click "Save & Run All (Commit)"** — NOT the cell play button.
4. Expected wall time: **~1.5-2.5 h**.

## After it finishes

On the local machine:
```bash
python scripts/import_cloud_artifacts.py
python scripts/finalize_discover.py   # overlap reports + S4 manifest
```


In [ ]:
# 1. Environment: keep the big HF cache OFF /kaggle/working (20 GB output cap)
import os

os.environ['HF_HOME'] = '/root/.cache/huggingface'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOGGING_MIN_SEVERITY'] = '3'
os.environ['PYTHONUNBUFFERED'] = '1'

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -g | head -2
!df -h /kaggle/working | tail -1


In [ ]:
# 2. Fresh code: clone + force tracked files to origin/main, then install
import os

%cd /kaggle/working
if not os.path.exists('/kaggle/working/circuit-graft'):
    !git clone --depth 1 https://github.com/0dvsh0/circuit-graft.git /kaggle/working/circuit-graft

%cd /kaggle/working/circuit-graft
!git fetch origin main
!git reset --hard origin/main
!git log --oneline -1

!pip uninstall -y torchao cudf-cu13 bigframes > /dev/null 2>&1 || true
!pip install --upgrade pip > /dev/null
!pip install --no-warn-conflicts safetune==0.1.12 circuitkit==0.2.0 auditkit==1.1.3 "curatorkit[generation,hf]==0.1.22" > /dev/null
!pip install --no-warn-conflicts "transformers==5.22.1" "bitsandbytes==0.50.2" "peft==0.21.2" "trl==1.14.2" "transformer-lens==3.8.0" "datasets==5.1.0" "accelerate==1.15.0" "huggingface-hub==1.33.0" sentence_transformers litellm python-dotenv typer rich > /dev/null
!pip uninstall -y torchao > /dev/null 2>&1 || true
!pip install -e . --no-deps > /dev/null
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao still installed'
import peft, transformers, torch
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 3. Auth: Kaggle secrets ONLY (no prompts - this is a headless commit run)
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'HF_TOKEN not found. Add-ons > Secrets > add HF_TOKEN (attach to this '
        'notebook), then Save & Run All again.'
    ) from exc

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_ORG'] = 'dubeydevansh'

from huggingface_hub import HfApi
api = HfApi(token=hf_token)
try:
    files = api.list_repo_files('CohereLabs/tiny-aya-global', repo_type='model')
    print('gated model access OK,', len(files), 'files')
except Exception as exc:
    raise RuntimeError(
        'tiny-aya-global is BLOCKED for this token. Accept the licence on the '
        'HF website with the SAME account that owns HF_TOKEN, then re-run.'
    ) from exc


In [ ]:
# 4. Data: import the verified snapshot (pairs for discovery; no generation)
import json
import os
from pathlib import Path

from huggingface_hub import snapshot_download

DATA_REPO = os.environ.get('CG_DATA_REPO', 'dubeydevansh/circuitgraft-phase4-data')
summary_path = Path('artifacts/data/summary.json')
if not summary_path.exists():
    snapshot_download(repo_id=DATA_REPO, repo_type='dataset', local_dir='artifacts/data',
                      token=os.environ['HF_TOKEN'])
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('data run_id:', summary.get('run_id'), '| G1:', summary.get('gate_g1', {}).get('status'))
assert Path('artifacts/data/pairs/safety.jsonl').exists(), 'safety pairs missing'
assert Path('artifacts/data/pairs/domain.jsonl').exists(), 'domain pairs missing'


In [ ]:
# 5. Pull the DRIFTED checkpoint from the cloud repo (uploaded by the local
# machine right after S2 finished - this run must not start without it).
import os
from pathlib import Path

from huggingface_hub import snapshot_download

CLOUD_REPO = 'dubeydevansh/circuitgraft-phase4-cloud'
merged_local = Path('artifacts/drift/scenarioA_seed0/merged')
if not (merged_local / 'config.json').exists():
    snapshot_download(
        repo_id=CLOUD_REPO,
        repo_type='dataset',
        local_dir='artifacts/drift',
        token=os.environ['HF_TOKEN'],
        allow_patterns=['artifacts/drift/scenarioA_seed0/merged/*'],
    )
files = sorted(p.name for p in merged_local.iterdir())
print('merged checkpoint files:', files)
assert 'config.json' in files and any(f.endswith('.safetensors') for f in files), (
    'merged drifted checkpoint incomplete - has the local machine finished '
    'upload_drift_to_hf.py?'
)


In [ ]:
# 7. S4 DOMAIN circuit discovery on the DRIFTED model (pulled from the cloud
# repo). Same canonical-directory scheme as the safety run.
import json
from pathlib import Path

from circuitgraft.config import BaseConfig, DiscoverConfig, load_config
from circuitgraft.discover.circuitkit_runner import (
    _model_ref,
    _repo_path,
    run_circuitkit_discovery,
)
from circuitgraft.graft.pipeline import load_layout
from circuitgraft.registry import config_hash

cfg = load_config(Path('configs/discover.yaml'), DiscoverConfig)
base = load_config(Path('configs/base.yaml'), BaseConfig)
run_id = f"discover-{config_hash(cfg)}"
out_dir = _repo_path(cfg.output_dir) / run_id / 'domain'
model_ref = _model_ref(base, cfg.domain.model)   # 'drifted' -> local merged checkpoint
print('run_id:', run_id)
print('model:', model_ref, '| kind: domain | ig_steps:', cfg.domain.ig_steps,
      '| n_examples:', cfg.domain.n_examples, '| batch:', cfg.domain.batch_size,
      '| seq:', cfg.domain.max_seq_len, '| dtype:', cfg.dtype)

result = run_circuitkit_discovery(
    model_ref,
    _repo_path(cfg.domain.pairs),
    out_dir,
    layout=load_layout(base.models.base),
    kind='domain',
    algorithm=cfg.domain.method,
    level=cfg.domain.granularity,
    n_examples=cfg.domain.n_examples,
    sparsity=cfg.domain.sparsity,
    ig_steps=cfg.domain.ig_steps,
    batch_size=cfg.domain.batch_size,
    max_seq_len=cfg.domain.max_seq_len,
    device=cfg.device,
    dtype=cfg.dtype,
    seed=cfg.seed,
    faithfulness=cfg.faithfulness,
)
payload = {k: v for k, v in result.items() if k != 'scores'}
print(json.dumps(payload, indent=2, default=str))
assert result['num_selected_nodes'] > 0, 'no nodes selected - refusing to treat as success'
print('FAITHFULNESS_ERROR:', result['faithfulness_error'])


In [ ]:
# 8. Verify the circuit artifacts on disk before uploading
import json
from pathlib import Path

from circuitgraft.config import BaseConfig, DiscoverConfig, load_config
from circuitgraft.registry import config_hash

cfg = load_config(Path('configs/discover.yaml'), DiscoverConfig)
base = load_config(Path('configs/base.yaml'), BaseConfig)
run_id = f"discover-{config_hash(cfg)}"
kind = 'domain'
circuit_dir = Path('artifacts/discover') / run_id / kind
required = ['circuit.pt', 'scores.json', 'circuit_info.json', 'selected.json']
for name in required:
    p = circuit_dir / name
    assert p.exists(), f'missing {name}'
    print(f'  {name:20s} {p.stat().st_size / 2**20:8.2f} MB')
info = json.loads((circuit_dir / 'circuit_info.json').read_text(encoding='utf-8'))
print('kind:', info.get('kind'), '| model:', info.get('model'))
if (circuit_dir / 'faithfulness.json').exists():
    print('faithfulness.json: present')
print('circuit dir:', circuit_dir)


In [ ]:
# 9. UPLOAD the circuit NOW - a later failure cannot destroy this
import os
from pathlib import Path

from circuitgraft.config import BaseConfig, DiscoverConfig, load_config
from huggingface_hub import HfApi
from circuitgraft.registry import config_hash

cfg = load_config(Path('configs/discover.yaml'), DiscoverConfig)
base = load_config(Path('configs/base.yaml'), BaseConfig)
run_id = f"discover-{config_hash(cfg)}"
kind = 'domain'

CLOUD_REPO = 'dubeydevansh/circuitgraft-phase4-cloud'
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(CLOUD_REPO, repo_type='dataset', private=True, exist_ok=True)
circuit_dir = Path('artifacts/discover') / run_id / kind
api.upload_folder(folder_path=str(circuit_dir), repo_id=CLOUD_REPO, repo_type='dataset',
                  path_in_repo=str(circuit_dir).replace('\\', '/'),
                  commit_message=f'cloud S4: {kind} circuit (ig_steps {cfg.safety.ig_steps}, cross_check off)')
print(f'{kind} circuit uploaded -> https://huggingface.co/datasets/' + CLOUD_REPO)
print()
print('On the local machine after this finishes:')
print('  python scripts/import_cloud_artifacts.py')
print('  python scripts/finalize_discover.py   # overlap + manifest once BOTH circuits are local')
